In [ ]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, monotonically_increasing_id, row_number, rand, lit
from pyspark.sql.window import Window

os.makedirs("/workspace/outputs/ncf", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("05_Prepare_NCF_Dataset")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

HDFS_GOLD = "hdfs://namenode:8020/netflix-recsys/gold"

interactions = spark.read.parquet(f"{HDFS_GOLD}/labeled_interactions")

In [ ]:
positive = (
    interactions
    .filter(col("label") == 1)
    .select("userId", "movieId", "label")
)

In [ ]:
# Lấy tối đa 2 triệu positive để TensorFlow chạy vừa tài nguyên.
positive_sample = positive.orderBy(rand(seed=42)).limit(2_000_000)

In [ ]:
users = (
    positive_sample
    .select("userId")
    .distinct()
    .orderBy("userId")
    .withColumn("user_idx", row_number().over(Window.orderBy("userId")) - 1)
)

movies = (
    positive_sample
    .select("movieId")
    .distinct()
    .orderBy("movieId")
    .withColumn("movie_idx", row_number().over(Window.orderBy("movieId")) - 1)
)

positive_idx = (
    positive_sample
    .join(users, "userId")
    .join(movies, "movieId")
    .select("user_idx", "movie_idx", "label")
)

In [ ]:
num_pos = positive_idx.count()
num_users = users.count()
num_movies = movies.count()

user_idx_df = users.select("user_idx")
movie_idx_df = movies.select("movie_idx")

negative = (
    user_idx_df.crossJoin(movie_idx_df)
    .join(
        positive_idx.select("user_idx", "movie_idx").withColumn("exists", lit(1)),
        ["user_idx", "movie_idx"],
        "left"
    )
    .filter(col("exists").isNull())
    .drop("exists")
    .orderBy(rand(seed=42))
    .limit(num_pos)
    .withColumn("label", lit(0))
)

In [ ]:
from pyspark.sql.functions import floor

negative_fast = (
    positive_idx
    .select("user_idx")
    .withColumn("movie_idx", floor(rand(seed=42) * num_movies).cast("int"))
    .withColumn("label", lit(0))
)

ncf_data = positive_idx.unionByName(negative_fast).orderBy(rand(seed=42))

In [ ]:
ncf_train, ncf_test = ncf_data.randomSplit([0.8, 0.2], seed=42)

In [ ]:
ncf_train.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/train_csv"
)

ncf_test.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/test_csv"
)

users.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/user_mapping_csv"
)

movies.coalesce(1).write.mode("overwrite").option("header", True).csv(
    "file:///workspace/outputs/ncf/movie_mapping_csv"
)

In [ ]:
metadata = {
    "num_users": num_users,
    "num_movies": num_movies,
    "num_positive_samples": num_pos,
    "negative_sampling_ratio": 1,
    "label_rule": "rating >= 4.0 => positive"
}

with open("/workspace/outputs/ncf/ncf_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

metadata